# T1CViz Tutorial: Visualization

Interactive graph visualization, event processing, and pattern detection.

```bash
pip install t1c-talon
```

In [ ]:
import os, numpy as np
from talon import ir, viz
os.makedirs("viz", exist_ok=True)
print(f"T1CViz: {viz.__version__}, Tonic: {viz.TONIC_AVAILABLE}")

## 1. Graph Visualization

In [ ]:
nodes = {
    "input": ir.Input(np.array([784])),
    "fc1": ir.Affine(weight=np.random.randn(256, 784).astype(np.float32)*0.01, bias=np.zeros(256, dtype=np.float32)),
    "lif1": ir.LIF(tau=np.ones(256, dtype=np.float32)*10, r=np.ones(256, dtype=np.float32), v_leak=np.zeros(256, dtype=np.float32), v_threshold=np.ones(256, dtype=np.float32)),
    "fc2": ir.Affine(weight=np.random.randn(10, 256).astype(np.float32)*0.01, bias=np.zeros(10, dtype=np.float32)),
    "lif2": ir.LIF(tau=np.ones(10, dtype=np.float32)*10, r=np.ones(10, dtype=np.float32), v_leak=np.zeros(10, dtype=np.float32), v_threshold=np.ones(10, dtype=np.float32)),
    "output": ir.Output(np.array([10])),
}
edges = [("input","fc1"),("fc1","lif1"),("lif1","fc2"),("fc2","lif2"),("lif2","output")]
graph = ir.Graph(nodes=nodes, edges=edges)
print(f"Graph: {len(graph.nodes)} nodes")

In [ ]:
viz.export_html(graph, "viz/snn.html", title="Simple SNN")
print("Exported to viz/snn.html")

In [ ]:
data = viz.graph_to_dict(graph)
print(f"Keys: {list(data.keys())}")
print(f"Params: {data['summary']['total_params']:,}")

## 2. Events

In [ ]:
n = 5000
sz = (28, 28)
ev = np.zeros(n, dtype=[('x',np.int32),('y',np.int32),('t',np.float64),('p',np.int32)])
ev['x'] = np.random.randint(0, sz[0], n)
ev['y'] = np.random.randint(0, sz[1], n)
ev['t'] = np.sort(np.random.uniform(0, 100000, n))
ev['p'] = np.random.choice([-1, 1], n)
print(f"Events: {n}")

In [ ]:
frames = viz.events_to_frames(ev, sensor_size=sz, n_frames=25)
print(f"Frames: {frames.shape}")
raster = viz.events_to_raster(ev, n_neurons=784, time_bins=100)
print(f"Raster: {raster.shape}")
grid = viz.events_to_grid(ev, sensor_size=sz, n_bins=5)
print(f"Grid: {grid.shape}")

In [ ]:
viz.export_events_html(ev, "viz/events.html", sensor_size=sz, title="Events", n_frames=30)
print("Exported event viz")

## 3. Pattern Detection

In [ ]:
skip_n = {
    "input": ir.Input(np.array([64,14,14])),
    "conv1": ir.Conv2d(weight=np.random.randn(64,64,3,3).astype(np.float32)*0.01, bias=np.zeros(64, dtype=np.float32), stride=(1,1), padding=(1,1)),
    "lif1": ir.LIF(tau=np.ones(64, dtype=np.float32)*10, r=np.ones(64, dtype=np.float32), v_leak=np.zeros(64, dtype=np.float32), v_threshold=np.ones(64, dtype=np.float32)),
    "conv2": ir.Conv2d(weight=np.random.randn(64,64,3,3).astype(np.float32)*0.01, bias=np.zeros(64, dtype=np.float32), stride=(1,1), padding=(1,1)),
    "skip": ir.Skip(input_type={"input": np.array([64,14,14])}, skip_type="residual"),
    "lif2": ir.LIF(tau=np.ones(64, dtype=np.float32)*10, r=np.ones(64, dtype=np.float32), v_leak=np.zeros(64, dtype=np.float32), v_threshold=np.ones(64, dtype=np.float32)),
    "output": ir.Output(np.array([64,14,14])),
}
skip_e = [("input","conv1"),("conv1","lif1"),("lif1","conv2"),("conv2","skip"),("input","skip"),("skip","lif2"),("lif2","output")]
sg = ir.Graph(nodes=skip_n, edges=skip_e)
for p in viz.detect_all_patterns(sg):
    print(f"  {p.pattern_type}: {p.nodes}")